## 财务指标生成


我现在需要你帮我在聚宽中导出市值数据，导到 QLib 中进行因子挖掘。导出的 CSV 需满足以下要求：时间范围为2010年1月1日至2026年5月31日，股票池为中证500，具体股票列表已写在该文件中(`csi500_distinct.txt`)。文件里面存的代码格式是`SH600008`,你要自行转换为聚宽的那种股票代码格式。最后输出文件到`csi500_市值_20100101_20260531.csv`

聚宽 PIT 数据导出需求

  文件格式

  - 单个 CSV 文件，包含所有股票
  - UTF-8 编码，逗号分隔

  字段结构
  - date 日期
  - symbol 股票代码（qlib格式）
  - 市值字段1
  - 市值字段2
  - 市值字段3
  - ......


  需要导出的指标：
  - 市值数据（表名: valuation）

  数据说明

  1. 每条记录 = 一只股票 × 一个报告期 × 一个指标。同一只股票同一报告期可以有多个不同 field 的记录
  2. date 必须是实际披露日，不能用报告期推算。如果聚宽没有提供披露日，请留空
  3. period 统一用日期格式，如一季报 2021-03-31，半年报 2021-06-30，三季报 2021-09-30，年报 2021-12-31
  4. 包含 ____ 年至 ____ 年的全部报告期数据
  5. 覆盖范围：全部 A 股 / 沪深300 / 中证500（请圈选）

  示例
```csv
  symbol,date,period,value,field
  sh600519,2021-04-28,2021-03-31,0.1285,roeWa
  sh600519,2021-04-28,2021-03-31,280.5,revenue
  sz000001,2021-04-30,2021-03-31,0.0821,roeWa
  sz000001,2021-04-30,2021-03-31,42.3,revenue
```

## api 调用方法
get_valuation 获取多个标的在指定交易日范围内的市值表数据

from jqdata import *
get_valuation(security, start_date=None, end_date=None, fields=None, count=None)

获取多个标的在指定交易日范围内的市值表数据

参数
security: 标的code字符串列表或者单个标的字符串
end_date: 查询结束时间
start_date: 查询开始时间，不能与count共用
count: 表示往前查询每一个标的count个交易日的数据，如果期间标的停牌，则该标的返回的市值数据数量小于count
fields: 财务数据中市值表的字段，返回结果中总会包含code、day字段，可用字段如下：
字段	释义
code	股票代码 带后缀.XSHE/.XSHG
day	日期 取数据的日期
capitalization	总股本(万股)
circulating_cap	流通股本(万股)
market_cap	总市值(亿元)
circulating_market_cap	流通市值(亿元)
turnover_ratio	换手率(%)
pe_ratio	市盈率(PE, TTM)
pe_ratio_lyr	市盈率(PE)
pb_ratio	市净率(PB)
ps_ratio	市销率(PS, TTM)
pcf_ratio	市现率(PCF, 现金净流量TTM)
返回值
返回一个dataframe，索引默认是pandas的整数索引，返回的结果中总会包含code、day字段。
注意
每次最多返回5000条数据，更多数据需要根据标的或者时间分多次获取
不要获取当天的估值数据,pe/市值等依赖收盘价的指标是盘后更新的。
示例
from jqdata import *
# 传入单个标的
df1 = get_valuation('000001.XSHE', end_date="2019-11-18", count=3, fields=['capitalization', 'market_cap'])
print(df1)

# 传入多个标的
df2 = get_valuation(['000001.XSHE', '000002.XSHE'], end_date="2019-11-18", count=3, fields=['capitalization', 'market_cap'])
print(df2)

In [ ]:
import pandas as pd
import time
import gc
from jqdata import *
from IPython.display import clear_output

# ======================== 配置 ========================
INPUT_FILE  = 'csi500_distinct.txt'
OUTPUT_FILE = 'csi500_市值_20100101_20260531.csv'
START_DATE  = '2010-01-01'
END_DATE    = '2026-05-31'
BATCH_SIZE  = 10
FIELDS = [
    'capitalization', 'circulating_cap', 'market_cap', 'circulating_market_cap',
    'turnover_ratio', 'pe_ratio', 'pe_ratio_lyr', 'pb_ratio', 'ps_ratio', 'pcf_ratio'
]

# ======================== 进度组件 ========================
class ProgressDisplay:
    """专用的数据导出进度展示组件"""

    def __init__(self, total_batches, total_stocks):
        self.total_batches = total_batches
        self.total_stocks  = total_stocks
        self.completed     = 0
        self.rows          = 0
        self.start_time    = time.time()
        self.errors        = 0
        self._last_display = 0
        self.first_errors  = []

    def update(self, new_rows=0, error=False, error_msg=''):
        self.completed += 1
        self.rows += new_rows
        if error:
            self.errors += 1
            if len(self.first_errors) < 5:
                self.first_errors.append(error_msg)
        now = time.time()
        if now - self._last_display < 0.3 and self.completed < self.total_batches:
            return
        self._last_display = now
        self._render()

    def _render(self):
        elapsed = time.time() - self.start_time
        speed   = self.completed / elapsed if elapsed > 0 else 0
        eta     = (self.total_batches - self.completed) / speed if speed > 0 else 0
        pct     = self.completed / self.total_batches

        bar_len = 50
        filled  = int(bar_len * pct)
        bar     = '█' * filled + '▒' * (bar_len - filled)

        clear_output(wait=True)
        print(f"{'='*60}")
        print(f"  市值数据导出  |  股票: {self.total_stocks}  |  {START_DATE} ~ {END_DATE}")
        print(f"{'='*60}")
        print(f"  [{bar}] {pct:.1%}")
        print(f"  批次: {self.completed:,} / {self.total_batches:,}")
        print(f"  已写入: {self.rows:,} 行  |  错误: {self.errors}")
        if elapsed > 0:
            print(f"  速度: {speed:.1f} 批/秒  |  {self.rows/elapsed:.0f} 行/秒")
        print(f"  已用: {self._fmt(elapsed)}  |  剩余: {self._fmt(eta)}")
        if self.first_errors:
            print(f"  --- 错误示例 ---")
            for e in self.first_errors[-3:]:
                print(f"  {e}")
        print(f"{'='*60}")

    def finish(self):
        elapsed = time.time() - self.start_time
        clear_output(wait=True)
        print(f"{'='*60}")
        print(f"  导出完成!")
        print(f"{'='*60}")
        print(f"  总股票数: {self.total_stocks}")
        print(f"  总数据行: {self.rows:,}")
        print(f"  总批次数: {self.completed:,}")
        print(f"  错误数:   {self.errors}")
        print(f"  总耗时:   {self._fmt(elapsed)}")
        if elapsed > 0:
            print(f"  平均速度: {self.rows/elapsed:.0f} 行/秒")
        if self.first_errors:
            print(f"\n  --- 错误详情 ---")
            for i, e in enumerate(self.first_errors, 1):
                print(f"  [{i}] {e}")
        print(f"{'='*60}")

    @staticmethod
    def _fmt(seconds):
        if seconds < 60:
            return f"{seconds:.1f}s"
        m, s = divmod(int(seconds), 60)
        if m < 60:
            return f"{m}m {s}s"
        h, m = divmod(m, 60)
        return f"{h}h {m}m"

# ======================== 读取 & 转换股票代码 ========================
with open(INPUT_FILE, 'r') as f:
    raw_codes = [line.strip() for line in f if line.strip()]

def to_jq(raw):
    return f"{raw[2:]}.XSHG" if raw[:2] == 'SH' else f"{raw[2:]}.XSHE"

def to_qlib(jq):
    num, suf = jq.split('.')
    return f"sh{num}" if suf == 'XSHG' else f"sz{num}"

jq_codes = [to_jq(c) for c in raw_codes]
print(f"已加载 {len(jq_codes)} 只股票")

# ======================== API 连通性测试 ========================
print("正在测试 API 连通性...")
try:
    test_df = get_valuation('000001.XSHE', end_date='2020-01-02', count=2, fields=['market_cap'])
    print(f"API 正常! 返回 {len(test_df)} 行")
    del test_df
except Exception as e:
    print(f"API 测试失败: {type(e).__name__}: {e}")
    raise

# ======================== 先写 CSV 表头 ========================
pd.DataFrame(columns=['date', 'symbol'] + FIELDS).to_csv(OUTPUT_FILE, index=False, encoding='utf-8')

# ======================== 构建年份区间 ========================
year_ranges = []
for y in range(2010, 2027):
    ys = f"{y}-01-01"
    ye = f"{y}-12-31" if y < 2026 else END_DATE
    year_ranges.append((ys, ye))

num_batches = ((len(jq_codes) + BATCH_SIZE - 1) // BATCH_SIZE) * len(year_ranges)
progress = ProgressDisplay(num_batches, len(jq_codes))

# ======================== 逐批拉取 & 追加写入 ========================
for i in range(0, len(jq_codes), BATCH_SIZE):
    batch_codes   = jq_codes[i:i + BATCH_SIZE]
    symbol_map    = {c: to_qlib(c) for c in batch_codes}

    for ys, ye in year_ranges:
        try:
            df = get_valuation(batch_codes, start_date=ys, end_date=ye, fields=FIELDS)
            if df is not None and not df.empty:
                df['symbol'] = df['code'].map(symbol_map)
                df['day'] = pd.to_datetime(df['day']).dt.strftime('%Y-%m-%d')
                df = df.rename(columns={'day': 'date'}).drop(columns=['code'])
                df = df[['date', 'symbol'] + FIELDS]
                # 立即追加写入磁盘，不在内存中累积
                df.to_csv(OUTPUT_FILE, mode='a', header=False, index=False, encoding='utf-8')
                progress.update(new_rows=len(df))
                del df
            else:
                progress.update()
        except Exception as e:
            err_desc = f"{batch_codes[0]}~{batch_codes[-1]} | {ys}~{ye} | {type(e).__name__}: {e}"
            progress.update(error=True, error_msg=err_desc)

    # 每批股票结束后主动回收内存
    gc.collect()

progress.finish()

# ======================== 验证输出文件 ========================
print(f"\n验证文件: {OUTPUT_FILE}")
import os
fsize = os.path.getsize(OUTPUT_FILE) / (1024 * 1024)
print(f"文件大小: {fsize:.1f} MB")
tail_df = pd.read_csv(OUTPUT_FILE, encoding='utf-8')
print(f"总行数: {len(tail_df):,}")
print(f"股票数: {tail_df['symbol'].nunique()}")
print(f"日期范围: {tail_df['date'].min()} ~ {tail_df['date'].max()}")
display(tail_df.head())
del tail_df
gc.collect()
